In [ ]:
# Cell 1: imports + EDGAR_IDENTITY セットアップ + log filter + モジュールキャッシュクリア
import logging
import os
import sys
from pathlib import Path
import edgar  # noqa: E402
from notebook.FILING_NLP.pipeline import config  # noqa: E402

REPO_ROOT = Path.cwd()
while REPO_ROOT.name and not (REPO_ROOT / ".env").exists():
    if REPO_ROOT.parent == REPO_ROOT:
        break
    REPO_ROOT = REPO_ROOT.parent
print(f"REPO_ROOT: {REPO_ROOT}")

env_path = REPO_ROOT / ".env"
if env_path.exists() and not os.environ.get("EDGAR_IDENTITY"):
    for line in env_path.read_text().splitlines():
        line = line.strip()
        if line.startswith("EDGAR_IDENTITY="):
            os.environ["EDGAR_IDENTITY"] = (
                line.split("=", 1)[1].strip().strip('"').strip("'")
            )
            break

edgar.set_identity(os.environ["EDGAR_IDENTITY"])
print(f"EDGAR_IDENTITY: {os.environ['EDGAR_IDENTITY']}")


class _LegacyParserFilter(logging.Filter):
    def filter(self, record):
        return "falling back to legacy parser" not in record.getMessage()


logging.getLogger("edgar.core").addFilter(_LegacyParserFilter())

for name in ["httpx", "httpxthrottlecache", "httpcore", "edgar.documents"]:
    logging.getLogger(name).setLevel(logging.WARNING)

sys.path.insert(0, str(REPO_ROOT))

# モジュールキャッシュをクリア (pipeline モジュールの変更を反映)
for mod_name in list(sys.modules):
    if mod_name.startswith("notebook.FILING_NLP.pipeline"):
        del sys.modules[mod_name]


print(f"NAS_ROOT: {config.NAS_ROOT}")
print(f"NAS exists: {config.NAS_ROOT.exists()}")


REPO_ROOT: /Users/yukihata/Desktop/quants
EDGAR_IDENTITY: YH-05 youxitiancore@gmail.com
NAS_ROOT: /Volumes/personal_folder/Quants/FILING_NLP_v2
NAS exists: True


# Indices v1 Pipeline: 4 米国インデックス × ヒストリカル全期間

`universe_indices_v1.parquet` (4 米国インデックス union) + `membership_indices_v1.parquet`
を用いて、`--index-filter` で絞り込んだ CIK 集合に対し Strategy C
(DOM table 除外 + ヒューリスティック subsection + paragraph packing) を
2002 年以降の全 10-K / 10-Q に適用する。

長時間実行は CLI (`python -m notebook.FILING_NLP.pipeline.run_indices`) に委譲し、
本 notebook は **CLI ラッパー** として以下に役割を集約する。

- Cell 4: コピペで `nohup` 起動できる CLI コマンド文字列の表示
- Cell 5: `INDICES_V1_PROGRESS_PATH` を 30 秒間隔で poll する進捗確認
- Cell 6-9: 完走後の per-CIK parquet 集約と GICS セクター別品質統計

## ⚠️ 生存バイアスの明示 (dec-2026-05-25-104)

本パイプラインの universe は **2026/5/22 時点 SPX/SOX/RIY/RAY 構成銘柄スナップショット**
である。2002-2026 年の全期間ヒストリカルに対して chunking を適用するため、
過去にインデックス構成銘柄だったが現在は除外されている銘柄 (倒産・買収・除外) は
**完全に欠落** している。

このため本データセットは厳密には

> 「2026/5/22 時点 SPX/SOX/RIY/RAY 構成銘柄の長期ヒストリカル分析」

と位置付けるべきであり、過去構成銘柄を含む再構築は**将来の独立議論**とする
(act-2026-05-25-108 で連携)。

`run_id = "indices_v1"` は本生存バイアス前提でのスナップショットラン ID であり、
後続の retro-active universe 拡張時には別 `run_id` を採番する。


In [2]:
# Cell 3: universe + membership ロード + index_filter 絞り込み + CIK 件数表示
import pandas as pd

# 絞り込み対象 index ("in_spx" / "in_sox" / "in_riy" / "in_ray")
INDEX_FILTER = "in_spx"

UNIVERSE_PATH = config.UNIVERSE_INDICES_V1_PARQUET
MEMBERSHIP_PATH = config.MEMBERSHIP_INDICES_V1_PARQUET

print(f"universe path:    {UNIVERSE_PATH}")
print(f"membership path:  {MEMBERSHIP_PATH}")
print(f"universe exists:  {UNIVERSE_PATH.exists()}")
print(f"membership exists:{MEMBERSHIP_PATH.exists()}")

universe = pd.read_parquet(UNIVERSE_PATH)
membership = pd.read_parquet(MEMBERSHIP_PATH)

print(f"\nuniverse rows:    {len(universe):,}")
print(f"membership rows:  {len(membership):,}")
print(f"universe columns: {list(universe.columns)}")
print(f"membership cols:  {list(membership.columns)}")

# index_filter == True の CIK 抽出 + universe と inner join
if INDEX_FILTER not in membership.columns:
    raise ValueError(
        f"membership parquet に列 {INDEX_FILTER!r} がありません. "
        f"columns: {list(membership.columns)}"
    )

target_ciks = membership.loc[membership[INDEX_FILTER].astype(bool), ["cik"]]
target_ciks["cik"] = target_ciks["cik"].astype("int64")
sample = universe.merge(target_ciks, on="cik", how="inner").reset_index(drop=True)
sample["cik"] = sample["cik"].astype("int64")
if "ticker" in sample.columns:
    sample["ticker"] = sample["ticker"].astype(str)

print(f"\n=== {INDEX_FILTER} 絞り込み後 ===")
print(f"  CIK 件数: {len(sample):,}")
if "gics_sector" in sample.columns:
    print("\n  GICS sector 分布:")
    print(sample["gics_sector"].value_counts(dropna=False).to_string())
print("\n  サンプル (先頭 10 件):")
display_cols = [
    c for c in ["cik", "ticker", "gics_sector", "index_name"] if c in sample.columns
]
print(sample[display_cols].head(10).to_string(index=False))


universe path:    /Volumes/personal_folder/Quants/FILING_NLP_v2/universe/universe_indices_v1.parquet
membership path:  /Volumes/personal_folder/Quants/FILING_NLP_v2/index_membership/membership_indices_v1.parquet
universe exists:  False
membership exists:False


FileNotFoundError: [Errno 2] No such file or directory: '/Volumes/personal_folder/Quants/FILING_NLP_v2/universe/universe_indices_v1.parquet'

In [ ]:
# Cell 4: パラメータ + checkpoint 確認 + CLI 起動コマンド表示
import json

RUN_ID = "indices_v1"
WORKERS = 8
RATE_RPS = 5.0
RATE_BURST = 10
FLUSH_EVERY = 5

SECTIONS_DIR = config.SECTIONS_DIR / RUN_ID
CHUNKS_DIR = config.CHUNKS_DIR / RUN_ID
FILINGS_DIR = config.FILINGS_METADATA_DIR / RUN_ID
CHECKPOINT_PATH = config.INDICES_V1_PROGRESS_PATH
ERRORS_PATH = config.LOGS_DIR / f"{RUN_ID}_errors.jsonl"
LOG_PATH = config.LOGS_DIR / f"{RUN_ID}_run.log"
SUMMARY_PATH = config.LOGS_DIR / f"{RUN_ID}_summary.json"

print(f"run_id:        {RUN_ID}")
print(f"workers:       {WORKERS}")
print(f"rate_rps:      {RATE_RPS}")
print(f"rate_burst:    {RATE_BURST}")
print(f"flush_every:   {FLUSH_EVERY}")
print(f"sections_dir:  {SECTIONS_DIR}")
print(f"chunks_dir:    {CHUNKS_DIR}")
print(f"filings_dir:   {FILINGS_DIR}")
print(f"checkpoint:    {CHECKPOINT_PATH}")
print(f"errors:        {ERRORS_PATH}")
print(f"log:           {LOG_PATH}")

# 既存 checkpoint の確認
if CHECKPOINT_PATH.exists():
    cp = json.loads(CHECKPOINT_PATH.read_text())
    print(f"\n既存 checkpoint: {len(cp.get('completed', {})):,} CIK 完了 (resume 可能)")
else:
    print("\n新規 run (checkpoint なし)")

# CLI 起動コマンド (nohup でコピペ実行可能な形式)
print("\n" + "=" * 80)
print("CLI 起動コマンド (コピペで nohup 実行可能):")
print("=" * 80)
print(f"""
nohup uv run python -m notebook.FILING_NLP.pipeline.run_indices \\
  --run-id {RUN_ID} \\
  --universe {UNIVERSE_PATH} \\
  --membership {MEMBERSHIP_PATH} \\
  --index-filter {INDEX_FILTER} \\
  --workers {WORKERS} \\
  --rate-rps {RATE_RPS} \\
  --rate-burst {RATE_BURST} \\
  --flush-every {FLUSH_EVERY} \\
  > {LOG_PATH} 2>&1 &
""")
print("=" * 80)
print("起動後 PID は `jobs -l` または `ps -ef | grep run_indices` で確認.")
print("中断は kill <pid>. checkpoint から resume 可能.")


In [ ]:
# Cell 5: 進捗 poll (INDICES_V1_PROGRESS_PATH から completed CIK 数を 30 秒間隔で確認)
import time
from datetime import datetime

from tqdm.auto import tqdm

POLL_INTERVAL_SEC = 30
TOTAL_CIK = len(sample)


def _load_completed_count(checkpoint_path: Path) -> int:
    """checkpoint JSON から completed CIK 数を読む. 未存在は 0."""
    if not checkpoint_path.exists():
        return 0
    try:
        cp = json.loads(checkpoint_path.read_text())
    except (json.JSONDecodeError, OSError):
        return 0
    completed = cp.get("completed", {})
    if isinstance(completed, dict):
        return len(completed)
    if isinstance(completed, list):
        return len(completed)
    return 0


print(f"total CIK:        {TOTAL_CIK:,}")
print(f"checkpoint:       {CHECKPOINT_PATH}")
print(f"poll interval:    {POLL_INTERVAL_SEC} 秒")
print("\nCtrl-C で poll 停止 (CLI 側は継続).")

start = time.time()
last_count = _load_completed_count(CHECKPOINT_PATH)
pbar = tqdm(total=TOTAL_CIK, initial=last_count, desc="indices_v1", unit="CIK")
try:
    while last_count < TOTAL_CIK:
        time.sleep(POLL_INTERVAL_SEC)
        new_count = _load_completed_count(CHECKPOINT_PATH)
        delta = new_count - last_count
        if delta > 0:
            pbar.update(delta)
            last_count = new_count
        # 1 イテレーション毎に最終更新時刻を表示
        elapsed = time.time() - start
        pbar.set_postfix(
            {
                "completed": f"{last_count:,}/{TOTAL_CIK:,}",
                "elapsed_min": f"{elapsed / 60:.1f}",
                "last_poll": datetime.now().strftime("%H:%M:%S"),
            }
        )
        if last_count >= TOTAL_CIK:
            break
except KeyboardInterrupt:
    print("\npoll 停止 (CLI 側は継続中).")
finally:
    pbar.close()

print(f"\n最終 completed: {last_count:,}/{TOTAL_CIK:,}")


In [ ]:
# Cell 6: per-CIK parquet 集約 (chunks_cik*.parquet を glob + concat)


def _load_dir(dir_path: Path, glob: str = "*.parquet") -> pd.DataFrame:
    """ディレクトリ配下の parquet を glob + concat して 1 DataFrame に."""
    files = sorted(dir_path.glob(glob))
    if not files:
        return pd.DataFrame()
    return pd.concat([pd.read_parquet(f) for f in files], ignore_index=True)


all_chunks = _load_dir(CHUNKS_DIR, glob="chunks_cik*.parquet")
if all_chunks.empty:
    # フォールバック: glob 命名規則が異なる場合は全 parquet を対象に
    all_chunks = _load_dir(CHUNKS_DIR)
all_sections = _load_dir(SECTIONS_DIR)
all_filings = _load_dir(FILINGS_DIR)

print("=== per-CIK 出力ファイル数 ===")
print(
    f"  sections files: {len(list(SECTIONS_DIR.glob('*.parquet'))):,} "
    f"(total {len(all_sections):,} rows)"
)
print(
    f"  chunks files:   {len(list(CHUNKS_DIR.glob('*.parquet'))):,} "
    f"(total {len(all_chunks):,} rows)"
)
print(
    f"  filings files:  {len(list(FILINGS_DIR.glob('*.parquet'))):,} "
    f"(total {len(all_filings):,} rows)"
)

print("\n=== ユニーク銘柄 ===")
if len(all_chunks):
    print(f"  unique CIK in chunks: {all_chunks['cik'].nunique():,}")
    print(f"  unique ticker:        {all_chunks['ticker'].nunique():,}")

print("\n=== fiscal_year 範囲 ===")
if len(all_chunks):
    print(f"  {all_chunks['fiscal_year'].min()} - {all_chunks['fiscal_year'].max()}")

print("\n=== token_count 統計 ===")
if len(all_chunks):
    print(all_chunks["token_count"].describe().round(1).to_string())
    over = (all_chunks["token_count"] > config.MAX_TOKENS).sum()
    if len(all_chunks) > 0:
        print(
            f"  上限超過 (>{config.MAX_TOKENS}): {over:,} "
            f"({100 * over / len(all_chunks):.2f}%)"
        )


In [ ]:
# Cell 7: filings 取得状況 / DOM 成功率 (item_1 / item_1a / item_2 / item_7) / chunks ゼロ CIK 分析
print("=== filings 取得状況 by status ===")
if len(all_filings):
    print(all_filings["status"].value_counts().to_string())

print("\n=== DOM section 取得成功率 by section_key ===")
if len(all_sections):
    dom_stats = all_sections.groupby("section_key").agg(
        n=("filing_id", "count"),
        dom_found=("dom_section_found", "sum"),
        tables_removed_mean=("tables_removed", "mean"),
        text_len_mean=("char_count", "mean"),
    )
    dom_stats["dom_rate_pct"] = (dom_stats["dom_found"] / dom_stats["n"] * 100).round(1)
    print(dom_stats.round(1).to_string())

print("\n=== chunks ゼロ CIK 分析 ===")
if len(all_chunks):
    cik_chunks = all_chunks.groupby(["cik", "ticker"]).size().to_frame("n_chunks")
else:
    cik_chunks = pd.DataFrame(columns=["n_chunks"])

sample_with_chunks = sample.merge(
    cik_chunks, left_on="cik", right_on="cik", how="left"
).fillna({"n_chunks": 0})
zero_chunk = sample_with_chunks[sample_with_chunks["n_chunks"] == 0]
print(
    f"  chunks ゼロ CIK 数: {len(zero_chunk):,} / {len(sample):,} "
    f"({100 * len(zero_chunk) / len(sample):.1f}%)"
)
if len(zero_chunk):
    display_cols = [
        c
        for c in ["cik", "ticker", "gics_sector", "index_name"]
        if c in zero_chunk.columns
    ]
    print("\n  先頭 20 件:")
    print(zero_chunk[display_cols].head(20).to_string(index=False))

print("\n=== errors.jsonl 集計 ===")
if ERRORS_PATH.exists():
    errs = []
    with ERRORS_PATH.open() as fp:
        for line in fp:
            try:
                errs.append(json.loads(line))
            except json.JSONDecodeError:
                continue
    print(f"  total errors logged: {len(errs):,}")
    if errs:
        phase_dist = pd.Series([e.get("phase", "unknown") for e in errs]).value_counts()
        print(f"  phase 分布:\n{phase_dist.to_string()}")
else:
    print("  (no errors logged)")


In [ ]:
# Cell 8: ticker × fiscal_year pivot 集計
if len(all_chunks):
    pivot = all_chunks.pivot_table(
        index="ticker",
        columns="fiscal_year",
        values="chunk_idx",
        aggfunc="count",
        fill_value=0,
    )
    print(f"ticker × fiscal_year pivot: shape={pivot.shape}")
    print("\n=== 先頭 20 ticker ===")
    print(pivot.head(20).to_string())

    print("\n=== fiscal_year 別 chunks 合計 ===")
    print(all_chunks.groupby("fiscal_year").size().to_string())

    print("\n=== form × section_key 分布 ===")
    print(
        all_chunks.groupby(["form", "section_key", "section_role"])
        .size()
        .to_frame("chunks")
        .to_string()
    )
else:
    print("(chunks が空のため pivot をスキップ)")


In [ ]:
# Cell 9: GICS セクター別 chunks / embedding 分布 (membership + universe join)
# 目的: セクター間でカバレッジ / 品質に偏りが無いかを定量チェック.
#   - per-CIK の平均 chunks 数
#   - chunks ゼロ CIK の比率 (取得失敗バイアス)
#   - chunks 合計の絶対量

if "gics_sector" not in sample.columns:
    print("⚠️ universe に gics_sector 列が無いため、セクター別分布をスキップします.")
else:
    sector_universe = sample.groupby("gics_sector").size().to_frame("n_cik_universe")

    if len(all_chunks):
        # 銘柄 -> セクター マッピング (universe から)
        cik_sector = sample[["cik", "gics_sector"]].drop_duplicates()
        chunks_with_sector = all_chunks.merge(cik_sector, on="cik", how="left")

        sector_chunks = chunks_with_sector.groupby("gics_sector").agg(
            n_chunks=("chunk_idx", "count"),
            n_cik_with_chunks=("cik", "nunique"),
        )
        sector_stats = sector_universe.join(sector_chunks, how="left").fillna(0)
        sector_stats["n_cik_with_chunks"] = sector_stats["n_cik_with_chunks"].astype(
            int
        )
        sector_stats["n_chunks"] = sector_stats["n_chunks"].astype(int)
        sector_stats["n_cik_zero"] = (
            sector_stats["n_cik_universe"] - sector_stats["n_cik_with_chunks"]
        )
        sector_stats["zero_pct"] = (
            100 * sector_stats["n_cik_zero"] / sector_stats["n_cik_universe"]
        ).round(1)
        sector_stats["chunks_per_cik_mean"] = (
            sector_stats["n_chunks"]
            / sector_stats["n_cik_with_chunks"].replace(0, pd.NA)
        ).round(1)

        sector_stats = sector_stats.sort_values("n_cik_universe", ascending=False)

        print("=== GICS セクター別 chunks / カバレッジ統計 ===")
        print(sector_stats.to_string())

        # セクター × form 別 chunks 分布 (深掘り)
        print("\n=== GICS sector × form 別 chunks ===")
        sector_form = (
            chunks_with_sector.groupby(["gics_sector", "form"])
            .size()
            .unstack(fill_value=0)
        )
        print(sector_form.to_string())
    else:
        # chunks が空の場合は universe 側のみ表示
        sector_universe["n_chunks"] = 0
        sector_universe["zero_pct"] = 100.0
        print("=== GICS セクター別 (universe のみ, chunks 未取得) ===")
        print(sector_universe.to_string())
